In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.optim.lr_scheduler import StepLR
from torch.utils.data import DataLoader, TensorDataset
import numpy as np
from sklearn.metrics import precision_score, recall_score, f1_score

from perforatedai import globals_perforatedai as GPA
from perforatedai import utils_perforatedai as UPA

# ==========================================================
# CONFIG
# ==========================================================

DATA_PATH = "../data/processed/modis_firms_train_val_test_dataset.npz"

BATCH_SIZE = 128
TEST_BATCH_SIZE = 512
EPOCHS = 20
LR = 0.001
WEIGHT_DECAY = 0.0
GAMMA = 0.95

MAX_TRAIN = 20000
MAX_VAL = 5000
MAX_TEST = 5000

device = "cpu"
print(f"[DEBUG] Using device: {device}")

np.random.seed(0)
torch.manual_seed(0)

# ==========================================================
# MODEL
# ==========================================================

class FireNN(nn.Module):
    def __init__(self, input_dim, hidden_dim=64):
        super().__init__()
        self.fc1 = nn.Linear(input_dim, hidden_dim)
        self.fc2 = nn.Linear(hidden_dim, hidden_dim)
        self.fc3 = nn.Linear(hidden_dim, 1)

    def forward(self, x):
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        x = self.fc3(x)  
        return x

# ==========================================================
# LOAD DATA
# ==========================================================

print("[DEBUG] Loading dataset...")

with np.load(DATA_PATH) as f:
    X_train = f["X_train"]
    y_train = f["y_train"]
    X_val = f["X_val"]
    y_val = f["y_val"]
    X_test = f["X_test"]
    y_test = f["y_test"]

print(f"[DEBUG] Original Train Size: {len(X_train):,}")

# ==========================================================
# BALANCE TRAINING DATA ONLY
# ==========================================================

def balanced_sample(X, y, max_samples):
    pos_idx = np.where(y == 1)[0]
    neg_idx = np.where(y == 0)[0]

    n_each = max_samples // 2
    n_each = min(n_each, len(pos_idx), len(neg_idx))

    pos_sample = np.random.choice(pos_idx, n_each, replace=False)
    neg_sample = np.random.choice(neg_idx, n_each, replace=False)

    idx = np.concatenate([pos_sample, neg_sample])
    np.random.shuffle(idx)

    return X[idx], y[idx]

def subsample_training(X, y, max_negatives=400_000):
    """Keep all positives, subsample negatives to max_negatives"""
    pos_idx = np.where(y == 1)[0]
    neg_idx = np.where(y == 0)[0]
    np.random.shuffle(neg_idx)
    neg_idx_sub = neg_idx[:max_negatives]

    idx = np.concatenate([pos_idx, neg_idx_sub])
    np.random.shuffle(idx)

    return X[idx], y[idx]

X_train, y_train = subsample_training(X_train, y_train, max_negatives=MAX_TRAIN//2)

# Keep validation and test balanced
X_val, y_val = balanced_sample(X_val, y_val, MAX_VAL)
X_test, y_test = balanced_sample(X_test, y_test, MAX_TEST)

print("[DEBUG] Train positive ratio:", np.mean(y_train))
print("[DEBUG] Val positive ratio:", np.mean(y_val))
print("[DEBUG] Test positive ratio:", np.mean(y_test))


# ==========================================================
# DATALOADERS
# ==========================================================

train_loader = DataLoader(
    TensorDataset(torch.tensor(X_train).float(),
                torch.tensor(y_train).float()),
    batch_size=BATCH_SIZE,
    shuffle=True
)

val_loader = DataLoader(
    TensorDataset(torch.tensor(X_val).float(),
                torch.tensor(y_val).float()),
    batch_size=TEST_BATCH_SIZE
)

test_loader = DataLoader(
    TensorDataset(torch.tensor(X_test).float(),
                torch.tensor(y_test).float()),
    batch_size=TEST_BATCH_SIZE
)

# ==========================================================
# LOSS FUNCTION WITH DYNAMIC pos_weight
# ==========================================================

pos_count = np.sum(y_train)
neg_count = len(y_train) - pos_count
pos_weight_tensor = torch.tensor([neg_count / pos_count], dtype=torch.float32).to(device)
loss_fn = nn.BCEWithLogitsLoss(pos_weight=pos_weight_tensor)
print(f"[DEBUG] pos_weight for BCE: {pos_weight_tensor.item():.2f}")

# ==========================================================
# TRAIN / EVAL
# ==========================================================

def train_epoch():
    model.train()
    correct = 0
    total_loss = 0

    for data, target in train_loader:
        data, target = data.to(device), target.to(device)

        optimizer.zero_grad()

        output = model(data)

        loss = loss_fn(output, target.unsqueeze(1))
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

        # logits threshold at 0
        preds = (output > 0).float()
        correct += preds.eq(target.unsqueeze(1)).sum().item()

    acc = 100.0 * correct / len(train_loader.dataset)
    return acc, total_loss / len(train_loader)


def evaluate(loader):
    model.eval()
    all_preds = []
    all_targets = []

    with torch.no_grad():
        for data, target in loader:
            data = data.to(device)
            output = model(data)

            # threshold logits at 0
            preds = (output > 0).cpu().numpy().flatten()

            all_preds.extend(preds)
            all_targets.extend(target.numpy())

    acc = np.mean(np.array(all_preds) == np.array(all_targets))
    precision = precision_score(all_targets, all_preds, zero_division=0)
    recall = recall_score(all_targets, all_preds, zero_division=0)
    f1 = f1_score(all_targets, all_preds, zero_division=0)

    return acc * 100, precision, recall, f1

# ==========================================================
# RUN EXPERIMENTS
# ==========================================================

for is_dendrite in [False, True]:

    print("\n" + "="*60)
    print(f"RUNNING WITH is_dendrite = {is_dendrite}")
    print("="*60)

    input_dim = X_train.shape[1]
    base_model = FireNN(input_dim)

    if is_dendrite:
        print("[DEBUG] Initializing WITH dendrites...")
        # model = UPA.initialize_pai(base_model, save_name="fire_model")

        GPA.pc.set_testing_dendrite_capacity(False)
        GPA.pc.set_weight_decay_accepted(True)
        GPA.pc.set_verbose(False)
        GPA.pc.set_switch_mode(GPA.pc.DOING_FIXED_SWITCH)
        GPA.pc.set_fixed_switch_num(4)
        
        model = UPA.initialize_pai(base_model, save_name="fire_model")
    
        GPA.pai_tracker.set_optimizer(optim.Adam)
        GPA.pai_tracker.set_scheduler(StepLR)
    else:
        print("[DEBUG] Running BASELINE...")
        model = base_model

    model.to(device)

    print(f"[DEBUG] Parameter count: {sum(p.numel() for p in model.parameters()):,}")

    if is_dendrite:
        optim_args = {"params": model.parameters(), "lr": LR, "weight_decay": WEIGHT_DECAY}
        sched_args = {"step_size": 1, "gamma": GAMMA}
        optimizer, scheduler = GPA.pai_tracker.setup_optimizer(model, optim_args, sched_args)
    else:
        optimizer = optim.Adam(model.parameters(), lr=LR)
        scheduler = StepLR(optimizer, step_size=1, gamma=GAMMA)

    # ---------------- TRAIN ----------------

    for epoch in range(1, EPOCHS + 1):

        train_acc, train_loss = train_epoch()
        scheduler.step()
        val_acc, val_prec, val_rec, val_f1 = evaluate(val_loader)

        print(f"Epoch {epoch}")
        print(f"  Train Acc: {train_acc:.2f}%")
        print(f"  Train Loss: {train_loss:.4f}")
        print(f"  Val Acc: {val_acc:.2f}%")
        print(f"  Precision: {val_prec:.4f}")
        print(f"  Recall:    {val_rec:.4f}")
        print(f"  F1 Score:  {val_f1:.4f}")

        if is_dendrite:
            model, restructured, training_complete = \
                GPA.pai_tracker.add_validation_score(val_f1, model)

            if restructured and not training_complete:
                print("[DEBUG] Restructured Reset optimizer")
                optim_args = {"params": model.parameters(), "lr": LR, "weight_decay": WEIGHT_DECAY}
                sched_args = {"step_size": 1, "gamma": GAMMA}
                optimizer, scheduler = GPA.pai_tracker.setup_optimizer(model, optim_args, sched_args)

            if training_complete:
                print("[DEBUG] Training ended early by PAI.")
                break


    # ---------------- TEST ----------------

    test_acc, test_prec, test_rec, test_f1 = evaluate(test_loader)

    print("\n========== FINAL TEST ==========")
    print(f"Test Acc: {test_acc:.2f}%")
    print(f"Precision: {test_prec:.4f}")
    print(f"Recall:    {test_rec:.4f}")
    print(f"F1 Score:  {test_f1:.4f}")

    if is_dendrite:
        print("Total dendrites added:",
            GPA.pai_tracker.member_vars.get("num_dendrites_added", 0))

Building dendrites without Perforated Backpropagation
[DEBUG] Using device: cpu
[DEBUG] Loading dataset...
[DEBUG] Original Train Size: 12,774,667
[DEBUG] Train positive ratio: 0.4926433282597666
[DEBUG] Val positive ratio: 0.5
[DEBUG] Test positive ratio: 0.5
[DEBUG] pos_weight for BCE: 1.03

RUNNING WITH is_dendrite = False
[DEBUG] Running BASELINE...
[DEBUG] Parameter count: 4,545
Epoch 1
  Train Acc: 53.29%
  Train Loss: 0.7071
  Val Acc: 56.94%
  Precision: 0.5533
  Recall:    0.7208
  F1 Score:  0.6260
Epoch 2
  Train Acc: 56.86%
  Train Loss: 0.6861
  Val Acc: 55.62%
  Precision: 0.5353
  Recall:    0.8530
  F1 Score:  0.6578
Epoch 3
  Train Acc: 57.69%
  Train Loss: 0.6845
  Val Acc: 56.70%
  Precision: 0.5469
  Recall:    0.7823
  F1 Score:  0.6437
Epoch 4
  Train Acc: 58.50%
  Train Loss: 0.6806
  Val Acc: 58.60%
  Precision: 0.5872
  Recall:    0.5790
  F1 Score:  0.5831
Epoch 5
  Train Acc: 57.88%
  Train Loss: 0.6819
  Val Acc: 59.03%
  Precision: 0.5761
  Recall:    0.684

/Users/amatthews/anaconda3/lib/python3.11/site-packages/torch/optim/lr_scheduler.py:143: UserWarning: Detected call of `lr_scheduler.step()` before `optimizer.step()`. In PyTorch 1.1.0 and later, you should call them in the opposite order: `optimizer.step()` before `lr_scheduler.step()`.  Failure to do this will result in PyTorch skipping the first value of the learning rate schedule. See more details at https://pytorch.org/docs/stable/optim.html#how-to-adjust-learning-rate
  warnings.warn("Detected call of `lr_scheduler.step()` before `optimizer.step()`. "


Epoch 7
  Train Acc: 58.34%
  Train Loss: 0.6818
  Val Acc: 59.54%
  Precision: 0.5827
  Recall:    0.6723
  F1 Score:  0.6243
Adding validation score 0.62427488
Checking PAI switch with mode n, switch mode DOING_FIXED_SWITCH, epoch 4, last improved epoch 4, total epochs 4, n: 4, num_cycles: 2
Returning False - learning rate optimization in progress. Not committed yet. Comparing initial 1 to last max 2
Epoch 8
  Train Acc: 58.57%
  Train Loss: 0.6786
  Val Acc: 59.35%
  Precision: 0.5921
  Recall:    0.6007
  F1 Score:  0.5964
Adding validation score 0.59637405
Checking PAI switch with mode n, switch mode DOING_FIXED_SWITCH, epoch 5, last improved epoch 4, total epochs 5, n: 4, num_cycles: 2
Returning False - learning rate optimization in progress. Not committed yet. Comparing initial 1 to last max 2
[DEBUG] Restructured Reset optimizer


/Users/amatthews/anaconda3/lib/python3.11/site-packages/torch/optim/lr_scheduler.py:143: UserWarning: Detected call of `lr_scheduler.step()` before `optimizer.step()`. In PyTorch 1.1.0 and later, you should call them in the opposite order: `optimizer.step()` before `lr_scheduler.step()`.  Failure to do this will result in PyTorch skipping the first value of the learning rate schedule. See more details at https://pytorch.org/docs/stable/optim.html#how-to-adjust-learning-rate
  warnings.warn("Detected call of `lr_scheduler.step()` before `optimizer.step()`. "


Epoch 9
  Train Acc: 58.54%
  Train Loss: 0.6809
  Val Acc: 58.82%
  Precision: 0.5764
  Recall:    0.6651
  F1 Score:  0.6176
Adding validation score 0.61758144
Checking PAI switch with mode n, switch mode DOING_FIXED_SWITCH, epoch 4, last improved epoch 4, total epochs 4, n: 4, num_cycles: 2
Returning False - learning rate optimization in progress. Not committed yet. Comparing initial 2 to last max 2
[DEBUG] Restructured Reset optimizer


...

Epoch 10
  Train Acc: 59.34%
  Train Loss: 0.6767
  Val Acc: 59.32%
  Precision: 0.5977
  Recall:    0.5704
  F1 Score:  0.5837
Adding validation score 0.58372265
Checking PAI switch with mode n, switch mode DOING_FIXED_SWITCH, epoch 6, last improved epoch 6, total epochs 6, n: 4, num_cycles: 2
Returning False - no triggers to switch have been hit
Epoch 11
  Train Acc: 59.13%
  Train Loss: 0.6763
  Val Acc: 59.75%
  Precision: 0.5797
  Recall:    0.7098
  F1 Score:  0.6382
Adding validation score 0.63815079
Checking PAI switch with mode n, switch mode DOING_FIXED_SWITCH, epoch 7, last improved epoch 7, total epochs 7, n: 4, num_cycles: 2
Returning True - Fixed switch number is hit
Importing best Model for switch to PA...
Switching back to N...
Resetting committed to initial rate to False
[DEBUG] Restructured Reset optimizer
Epoch 12
  Train Acc: 59.32%
  Train Loss: 0.6761
  Val Acc: 59.30%
  Precision: 0.5666
  Recall:    0.7910
  F1 Score:  0.6602
Adding validation score 0.66024870
C

/Users/amatthews/anaconda3/lib/python3.11/site-packages/torch/optim/lr_scheduler.py:143: UserWarning: Detected call of `lr_scheduler.step()` before `optimizer.step()`. In PyTorch 1.1.0 and later, you should call them in the opposite order: `optimizer.step()` before `lr_scheduler.step()`.  Failure to do this will result in PyTorch skipping the first value of the learning rate schedule. See more details at https://pytorch.org/docs/stable/optim.html#how-to-adjust-learning-rate
  warnings.warn("Detected call of `lr_scheduler.step()` before `optimizer.step()`. "


Epoch 14
  Train Acc: 58.89%
  Train Loss: 0.6763
  Val Acc: 59.56%
  Precision: 0.5750
  Recall:    0.7333
  F1 Score:  0.6446
Adding validation score 0.64456177
Checking PAI switch with mode n, switch mode DOING_FIXED_SWITCH, epoch 8, last improved epoch 8, total epochs 8, n: 4, num_cycles: 4
Returning False - learning rate optimization in progress. Not committed yet. Comparing initial 1 to last max 4
[DEBUG] Restructured Reset optimizer


/Users/amatthews/anaconda3/lib/python3.11/site-packages/torch/optim/lr_scheduler.py:143: UserWarning: Detected call of `lr_scheduler.step()` before `optimizer.step()`. In PyTorch 1.1.0 and later, you should call them in the opposite order: `optimizer.step()` before `lr_scheduler.step()`.  Failure to do this will result in PyTorch skipping the first value of the learning rate schedule. See more details at https://pytorch.org/docs/stable/optim.html#how-to-adjust-learning-rate
  warnings.warn("Detected call of `lr_scheduler.step()` before `optimizer.step()`. "


Epoch 15
  Train Acc: 59.39%
  Train Loss: 0.6757
  Val Acc: 59.44%
  Precision: 0.5926
  Recall:    0.6040
  F1 Score:  0.5983
Adding validation score 0.59828653
Checking PAI switch with mode n, switch mode DOING_FIXED_SWITCH, epoch 10, last improved epoch 10, total epochs 10, n: 4, num_cycles: 4
Returning False - no triggers to switch have been hit
Epoch 16
  Train Acc: 59.80%
  Train Loss: 0.6727
  Val Acc: 60.12%
  Precision: 0.5829
  Recall:    0.7112
  F1 Score:  0.6407
Adding validation score 0.64069264
Checking PAI switch with mode n, switch mode DOING_FIXED_SWITCH, epoch 11, last improved epoch 10, total epochs 11, n: 4, num_cycles: 4
Returning True - Fixed switch number is hit
Importing best Model for switch to PA...
Switching back to N...
Resetting committed to initial rate to False
[DEBUG] Restructured Reset optimizer
Epoch 17
  Train Acc: 59.40%
  Train Loss: 0.6750
  Val Acc: 60.31%
  Precision: 0.5862
  Recall:    0.7006
  F1 Score:  0.6384
Adding validation score 0.6383